In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

In [2]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip -q ml-1m.zip

In [3]:
ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    header=None,
    names=["userId", "movieId", "rating", "timestamp"],
    engine="python"
)

movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    header=None,
    names=["movieId", "title", "genres"],
    engine="python",
    encoding="latin-1"
)
# Map to contiguous indices
ratings["user"] = ratings.userId.astype("category").cat.codes
ratings["item"] = ratings.movieId.astype("category").cat.codes
user_categories = ratings.userId.astype("category").cat.categories
item_categories = ratings.movieId.astype("category").cat.categories

In [4]:
num_users = len(user_categories)
num_items = len(item_categories)
print(f"Loaded {len(ratings)} ratings, {num_users} users, {num_items} items")

Loaded 1000209 ratings, 6040 users, 3706 items


In [5]:
train_val, test_df = train_test_split(
    ratings, test_size=0.2, random_state=42
)
train_df, val_df   = train_test_split(
    train_val, test_size=0.1, random_state=42
)

print(f"Splits → train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

Splits → train: 720150, val: 80017, test: 200042


In [6]:
class RatingDataset(Dataset):
    def __init__(self, df):
        self.u = torch.tensor(df.user.values, dtype=torch.long)
        self.i = torch.tensor(df.item.values, dtype=torch.long)
        self.r = torch.tensor(df.rating.values, dtype=torch.float32)
    def __len__(self):
        return len(self.u)
    def __getitem__(self, idx):
        return self.u[idx], self.i[idx], self.r[idx]

train_dl = DataLoader(RatingDataset(train_df), batch_size=1024, shuffle=True)
val_dl   = DataLoader(RatingDataset(val_df),   batch_size=1024)
test_dl  = DataLoader(RatingDataset(test_df),  batch_size=1024)

In [7]:
#Two-Tower regression model
class UserTower(nn.Module):
    def __init__(self, U, D):
        super().__init__() #when usinng object oriented programming you just have to initialize this
        self.emb = nn.Embedding(U, D) #embedding of the number of users in dimension D
        self.mlp = nn.Sequential(nn.Linear(D, D), nn.ReLU()) #multilayer perceptron
    def forward(self, u): return self.mlp(self.emb(u)) #forward simply pushed the emdding through the network in one epoch

class ItemTower(nn.Module):
    def __init__(self, I, D):
        super().__init__()
        self.emb = nn.Embedding(I, D)
        self.mlp = nn.Sequential(nn.Linear(D, D), nn.ReLU())
    def forward(self, i): return self.mlp(self.emb(i))

class TwoTowerRegressor(nn.Module):
    def __init__(self, U, I, D=64):
        super().__init__()
        self.user_tower = UserTower(U, D)
        self.item_tower = ItemTower(I, D)
        # Initialize bias terms
        self.user_bias = nn.Embedding(U, 1)
        self.item_bias = nn.Embedding(I, 1)
        self.global_bias = nn.Parameter(torch.tensor([0.0]))

    def forward(self, u, i):
        u_vec = self.user_tower(u)   # gives user vector in 2D
        i_vec = self.item_tower(i)   # gives us item vector in 2D
        dot = (u_vec * i_vec).sum(dim=1)  # Dot product in 1D
        bias = self.user_bias(u).squeeze() + self.item_bias(i).squeeze() + self.global_bias
        return dot + bias

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # If the GPU i available run via cuda, if not retuen to CPU. GPU is faster and more efficient and free
model  = TwoTowerRegressor(num_users, num_items, D=64).to(device)
opt    = optim.Adam(model.parameters(), lr=1e-3) # we will be using Adaptive movement estimation because it adpats learning rate to each parameter, converges faster and works well with random default parameters
mse_loss = nn.MSELoss()

In [9]:
#Training and validation
EPOCHS = 10
for epoch in range(1, EPOCHS+1):
    # train
    model.train() # sets the model in training mode so that the gradients can be modifiable
    total_loss = 0.0 # Initializes the loss
    for u,i,r in train_dl:
        u,i,r = u.to(device), i.to(device), r.to(device) # Pushes these inputs to the GPU or CPU
        opt.zero_grad() # resets gradients
        preds = model(u, i)
        loss  = mse_loss(preds, r)
        loss.backward() # INitaites the backpropagation required for learning
        opt.step() # updates the weights and gradients
        total_loss += loss.item() * u.size(0)
    train_rmse = np.sqrt(total_loss / len(train_dl.dataset))

     # validate
    model.eval() #sets the model in evaluation mode
    val_preds, val_trues = [], []
    with torch.no_grad(): # we do not need gradients here as the model is not learning so they will not take up unnecessary space
        for u,i,r in val_dl:
            u,i = u.to(device), i.to(device)
            preds = model(u, i).cpu().numpy()
            val_preds.extend(preds)
            val_trues.extend(r.numpy())
    val_rmse = np.sqrt(mean_squared_error(val_trues, val_preds))
    val_mae  = mean_absolute_error(val_trues, val_preds)
    print(f"Epoch {epoch:2d}  Train RMSE={train_rmse:.4f}  Val RMSE={val_rmse:.4f}  Val MAE={val_mae:.4f}")

Epoch  1  Train RMSE=1.5277  Val RMSE=1.2289  Val MAE=0.9689
Epoch  2  Train RMSE=1.1223  Val RMSE=1.0655  Val MAE=0.8414
Epoch  3  Train RMSE=1.0132  Val RMSE=1.0019  Val MAE=0.7923
Epoch  4  Train RMSE=0.9655  Val RMSE=0.9707  Val MAE=0.7665
Epoch  5  Train RMSE=0.9406  Val RMSE=0.9540  Val MAE=0.7525
Epoch  6  Train RMSE=0.9278  Val RMSE=0.9432  Val MAE=0.7466
Epoch  7  Train RMSE=0.9197  Val RMSE=0.9375  Val MAE=0.7405
Epoch  8  Train RMSE=0.9148  Val RMSE=0.9358  Val MAE=0.7387
Epoch  9  Train RMSE=0.9111  Val RMSE=0.9334  Val MAE=0.7379
Epoch 10  Train RMSE=0.9080  Val RMSE=0.9336  Val MAE=0.7358


In [10]:
# Test evaluation on the test set
model.eval()
test_preds, test_trues = [], []
with torch.no_grad():
    for u,i,r in test_dl:
        u,i = u.to(device), i.to(device)
        preds = model(u, i).cpu().numpy()
        test_preds.extend(preds)
        test_trues.extend(r.numpy())
test_rmse = mean_squared_error(test_trues, test_preds)
test_mae  = mean_absolute_error(test_trues, test_preds)
print(f"\nTest RMSE={test_rmse:.4f}  Test MAE={test_mae:.4f}")


Test RMSE=0.8843  Test MAE=0.7430


In [11]:
#Recommendations by rating
def recommend(uid, k=10):
    u_idx = torch.tensor([uid], dtype=torch.long, device=device)
    with torch.no_grad():
        u_vec  = model.user_tower(u_idx) #gives us a user vector
        all_i  = torch.arange(num_items, dtype=torch.long, device=device)
        i_vec  = model.item_tower(all_i)      # gives us all the item vectors
        preds  = (u_vec * i_vec).sum(dim=1).cpu().numpy()   #returns the INDICES in a numpy array
    topk = np.argsort(preds)[-k:][::-1] # returns the top ten indices
    print(f"\nTop {k} for user {uid}:")
    for rank, idx in enumerate(topk, 1):
        raw_id = item_categories[idx]    # original movieId using the variable stored the item categoris earlier
        title  = movies.loc[movies.movieId==raw_id, "title"].values[0]
        print(f"{rank:2d}. {title:<50s}  pred_rating={preds[idx]:.2f}")

# demo
recommend(uid=3, k=10)


Top 10 for user 3:
 1. Scent of a Woman (1992)                             pred_rating=7.54
 2. Platoon (1986)                                      pred_rating=7.51
 3. Trouble in Paradise (1932)                          pred_rating=7.46
 4. Raiders of the Lost Ark (1981)                      pred_rating=7.45
 5. Limey, The (1999)                                   pred_rating=7.39
 6. Henry V (1989)                                      pred_rating=7.28
 7. Being There (1979)                                  pred_rating=7.19
 8. Broadcast News (1987)                               pred_rating=7.13
 9. Cyclo (1995)                                        pred_rating=7.12
10. Terminator, The (1984)                              pred_rating=7.11


In [12]:
import os


In [13]:
save_path = os.path.join(os.getcwd(), "two_tower_model.pt")
torch.save(model, save_path)
print(f"Saved to: {save_path}")

Saved to: /content/two_tower_model.pt


In [14]:
from google.colab import files
files.download("two_tower_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
shutil.make_archive("ml-1m", "zip", "ml-1m")
files.download("ml-1m.zip")

In [ ]:
import pickle

# Save item_categories separately
with open("item_categories.pkl", "wb") as f:
    pickle.dump(item_categories, f)

# Save movies dataframe separately
movies_df = movies[["movieId", "title", "genres"]]
movies_df.to_csv("movies.csv", index=False)

#files.download("two_tower_model.pt")
files.download("item_categories.pkl")
files.download("movies.csv")